In [22]:
%pip install -qU langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"

Note: you may need to restart the kernel to use updated packages.


In [23]:
import os
import shutil
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")

GITHUB_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")      # only needed for the GitHub server
SAMPLE_DOCS_DIR = os.path.join(os.getcwd(), "sample_docs")   # the ONLY folder the Filesystem server may see
# Using other servers? Load their secrets here the same way, e.g. NOTION_TOKEN = os.getenv("NOTION_TOKEN")

print("OpenAI key:   loaded")
print("Node.js npx: ", "found" if shutil.which("npx") else "NOT found: needed only for servers started with npx")
print("GitHub token:", "loaded" if GITHUB_TOKEN else "missing: needed only if you use the GitHub server")
print("sample_docs: ", "found" if os.path.isdir(SAMPLE_DOCS_DIR) else "NOT found: put sample_docs/ next to this notebook")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

OpenAI key:   loaded
Node.js npx:  found
GitHub token: loaded
sample_docs:  found


In [24]:
import sys

if sys.platform == "win32":
    import mcp.client.stdio as _stdio
    _stdio.stdio_client.__wrapped__.__defaults__ = (open(os.devnull, "w"),)

In [25]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

async def log_tool_calls(request, handler):
    # BEFORE: which server, which tool, which inputs
    print(f"[LOG] {request.server_name} -> {request.name}  {request.args}")
    result = await handler(request)   # the real tool runs on the server
    # AFTER: what the server sent back
    text = " ".join(getattr(part, "text", str(part)) for part in result.content)
    print(f"[LOG]    returned: {text[:150]}...")
    return result

In [26]:
read_only = {"list_directory", "read_text_file", "read_file"}
fs_read_tools = [t for t in fs_tools if t.name in read_only]
all_tools = fs_read_tools + github_tools

# STEP 2, MCP CONNECTING

In [27]:
# YOUR TURN: Filesystem client

# TODO: filesystem_client = MultiServerMCPClient({ "filesystem": { ... } }, tool_interceptors=[...])

In [28]:
# Step 2a: Filesystem client (stdio)
filesystem_client = MultiServerMCPClient(
    {
        "filesystem": {                       # the server's name (shows up in [LOG] lines)
            "transport": "stdio",             # the client starts the server on your laptop
            "command": "npx",                 # the program to run
            "args": [
                "-y",                                       # don't ask "install? y/n"
                "@modelcontextprotocol/server-filesystem",  # the server package
                SAMPLE_DOCS_DIR,                            # the ONLY folder it may see
            ],
        }
    },
    tool_interceptors=[log_tool_calls],       # plug in the logger
)

In [29]:
# YOUR TURN: GitHub client

# TODO: github_client = MultiServerMCPClient({ "github": { ... } }, tool_interceptors=[...])

In [30]:
# Step 2b: GitHub client (HTTP + token)
github_client = MultiServerMCPClient(
    {
        "github": {                                        # the server's name (shows up in [LOG] lines)
            "transport": "http",                           # the client CALLS a server on GitHub's machines
            "url": "https://api.githubcopilot.com/mcp/",   # GitHub's official MCP server (keep the final /)
            "headers": {
                "Authorization": f"Bearer {GITHUB_TOKEN}", # your token, read from .env in Step 1
                "X-MCP-Toolsets": "repos,issues",          # only load repo and issue tools
            },
        }
    },
    tool_interceptors=[log_tool_calls],                    # same logger as the filesystem client
)

In [31]:
# Step 2c: test both connections
for name, client in [("filesystem", filesystem_client), ("github", github_client)]:
    # Tools
    tools = await client.get_tools()
    print(f"{name}: {len(tools)} tools")

    # Resources (not every server offers them)
    try:
        resources = await client.get_resources(name)
        print(f"{name}: {len(resources)} resources")
    except Exception as e:
        print(f"{name}: no resources ({type(e).__name__})")

filesystem: 14 tools
filesystem: no resources (ExceptionGroup)
github: 29 tools
github: 4 resources


# STEP 3, LOAD TOOLS LANGCHAIN

In [32]:
# Step 3: load the tools from both clients
fs_tools = await filesystem_client.get_tools()
github_tools = await github_client.get_tools()

print(f"Filesystem tools ({len(fs_tools)}):")
for t in fs_tools:
    print("  -", t.name)

print(f"\nGitHub tools ({len(github_tools)}):")
for t in github_tools:
    print("  -", t.name)

Filesystem tools (14):
  - read_file
  - read_text_file
  - read_media_file
  - read_multiple_files
  - write_file
  - edit_file
  - create_directory
  - list_directory
  - list_directory_with_sizes
  - directory_tree
  - move_file
  - search_files
  - get_file_info
  - list_allowed_directories

GitHub tools (29):
  - add_issue_comment
  - create_branch
  - create_or_update_file
  - create_repository
  - delete_file
  - fork_repository
  - get_commit
  - get_file_contents
  - get_label
  - get_latest_release
  - get_release_by_tag
  - get_tag
  - issue_read
  - issue_write
  - list_branches
  - list_commits
  - list_issue_fields
  - list_issue_types
  - list_issues
  - list_releases
  - list_repository_collaborators
  - list_tags
  - push_files
  - search_code
  - search_commits
  - search_issues
  - search_repositories
  - sub_issue_write
  - update_issue_comment


# STEP 4, AGENT

In [33]:
# Step 4a: Filesystem agent
fs_agent = create_agent(
    model=llm,
    tools=fs_tools,
    system_prompt=(
        f"You answer questions about project documents stored in {SAMPLE_DOCS_DIR}. "
        "Always list that directory first, then read the relevant files with your tools. "
        "Answer only from what the files say. If the answer is not in the files, say so. "
        "Never guess. Name the file each fact comes from."
    ),
)

question = "Who is the engagement lead for Project Atlas, and what is the target go-live date?"

fs_result = await fs_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nANSWER:\n", fs_result["messages"][-1].content)

[LOG] filesystem -> list_directory  {'path': '/Users/pieteralbrecht/Desktop/IRONHACK/WEEK_4/WEEK_4_DAY_3/WEEK_4_DAY_3_LAB/sample_docs'}
[LOG]    returned: [FILE] faq.md
[FILE] projects_notes.md...
[LOG] filesystem -> read_text_file  {'path': '/Users/pieteralbrecht/Desktop/IRONHACK/WEEK_4/WEEK_4_DAY_3/WEEK_4_DAY_3_LAB/sample_docs/faq.md'}
[LOG] filesystem -> read_text_file  {'path': '/Users/pieteralbrecht/Desktop/IRONHACK/WEEK_4/WEEK_4_DAY_3/WEEK_4_DAY_3_LAB/sample_docs/projects_notes.md'}
[LOG]    returned: ...
[LOG]    returned: ...

ANSWER:
 The files do not contain any information regarding the engagement lead for Project Atlas or the target go-live date.


In [34]:
# Step 4b: GitHub agent
github_agent = create_agent(
    model=llm,
    tools=github_tools,
    system_prompt=(
        "You answer questions about GitHub repositories by reading their files with your GitHub tools. "
        "Answer only from what the files say. Never guess. "
        "Always name the repository and the file path you used."
    ),
)

question = (
    "Read the README of the GitHub repository langchain-ai/langchain-mcp-adapters. "
    "Which MCP transports does it show in its examples?"
)

github_result = await github_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nANSWER:\n", github_result["messages"][-1].content)

[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langchain-mcp-adapters', 'path': 'README.md'}
[LOG]    returned: successfully downloaded text file (SHA: 0d3e7d0fc7c4facdf437445eaeeb71102de4befe) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...

ANSWER:
 The README of the repository **langchain-ai/langchain-mcp-adapters** shows examples of the following MCP transports:

1. **stdio** - Used for a simple server that can add and multiply numbers.
2. **http** - Used for a weather server that retrieves weather information.
3. **streamable HTTP** - Mentioned as a transport that supports streamable connections.

These transports are demonstrated in the examples provided in the README.


In [35]:
# Step 5: resource -> system prompt
docs_client = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp",
        }
    }
)

resources = await docs_client.get_resources("langchain-docs")
print(f"{len(resources)} resources")

guide_text = resources[0].as_string()
print(f"Resource length: {len(guide_text)} characters")
print(guide_text[:500])   # peek at what you actually got

guide_agent = create_agent(
    model=llm,
    tools=[],   # no tools: the agent can only use what's in the prompt
    system_prompt=(
        "Answer ONLY from the guide below. If the guide does not cover the question, "
        "say 'not in the guide'.\n\n--- GUIDE ---\n" + guide_text
    ),
)

question = "What do I need to add to an agent so that it remembers the conversation between calls?"
guide_result = await guide_agent.ainvoke({"messages": [HumanMessage(content=question)]})
answer = guide_result["messages"][-1].content
print("\nANSWER:\n", answer)

1 resources
Resource length: 11807 characters
---
name: langchain
description: Use when building AI agents, integrating language models with
  tools, creating multi-step workflows, or developing applications that need to
  interact with external APIs and data sources. Reach for LangChain when you
  need a configurable agent harness with model integration, tool calling, memory
  management, and middleware support.
metadata:
  mintlify-proj: langchain
  version: "1.0"
---

# LangChain Skill

## Product summary

LangChain is a framework for bu

ANSWER:
 You need to pass a `thread_id` in the config to persist conversation history across invocations. Use `config={"configurable": {"thread_id": "..."}}` when invoking the agent. Additionally, ensure you have a checkpointer configured if persistence is needed.


In [36]:
# Evidence: do the key words from the answer appear in the resource?
for word in ["checkpointer", "thread_id"]:   # replace with words from YOUR answer
    print(f"{word!r} in resource: {word in guide_text}")

'checkpointer' in resource: True
'thread_id' in resource: True


In [37]:
# Step 6a: two clients -> one tool list
all_tools = fs_tools + github_tools

print(f"{len(fs_tools)} filesystem + {len(github_tools)} github = {len(all_tools)} tools")
print([t.name for t in all_tools])

14 filesystem + 29 github = 43 tools
['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories', 'add_issue_comment', 'create_branch', 'create_or_update_file', 'create_repository', 'delete_file', 'fork_repository', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'issue_write', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'push_files', 'search_code', 'search_commits', 'search_issues', 'search_repositories', 'sub_issue_write', 'update_issue_comment']


In [38]:
# Step 6b: combined agent (improved prompt)
combined_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You answer questions using exactly two sources. Use tools for every fact; never answer from memory.\n\n"

        "SOURCE 1: local project files (filesystem tools)\n"
        f"- The files live ONLY in this folder: {SAMPLE_DOCS_DIR}\n"
        "- Step 1: call list_directory on that exact folder path.\n"
        "- Step 2: read the file you need with read_text_file, using the FULL absolute path, "
        f"for example: {SAMPLE_DOCS_DIR}/project_notes.md\n"
        "- Never use a bare file name or a path outside that folder.\n"
        "- Do not use search_files to look for text: it only matches file names. Read the file instead.\n\n"

        "SOURCE 2: GitHub repositories (GitHub tools)\n"
        "- Read files with get_file_contents, giving owner, repo and path separately "
        "(e.g. owner='langchain-ai', repo='langchain-mcp-adapters', path='README.md').\n\n"

        "ANSWER RULES\n"
        "- Answer each part of the question separately.\n"
        "- After each claim, name its source: the file name, or repository + file path.\n"
        "- If a source does not contain the answer, say so explicitly. Do not fill gaps with your own knowledge.\n"
        "- If you add your own reasoning on top of the sources, label it clearly as 'Reasoning:'."
    ),
)

question = (
    "Read our Project Atlas notes (project_notes.md): what constraint do they set about where ticket data may live? "
    "Then read the README of the GitHub repository langchain-ai/langchain-mcp-adapters: "
    "which transport would let us run an MCP server inside the client's own environment, "
    "so that data never leaves it?"
)

combined_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nANSWER:\n", combined_result["messages"][-1].content)

[LOG] filesystem -> read_text_file  {'path': '/Users/pieteralbrecht/Desktop/IRONHACK/WEEK_4/WEEK_4_DAY_3/WEEK_4_DAY_3_LAB/sample_docs/project_notes.md'}
[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langchain-mcp-adapters', 'path': 'README.md'}
[LOG]    returned: ENOENT: no such file or directory, open '/Users/pieteralbrecht/Desktop/IRONHACK/WEEK_4/WEEK_4_DAY_3/WEEK_4_DAY_3_LAB/sample_docs/project_notes.md'...
[LOG]    returned: successfully downloaded text file (SHA: 0d3e7d0fc7c4facdf437445eaeeb71102de4befe) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...

ANSWER:
 1. **Constraint on Ticket Data Location**: I was unable to read the Project Atlas notes from the specified file path, as it appears the file does not exist. Therefore, I cannot provide information about any constraints set regarding where ticket data may live.

2. **Transport for Running MCP Server in Client's Environment**: The README of the GitHub repository `langchain-ai/la

---
## ✅ Write `lab_proof.md` [YOUR TURN]

Create `lab_proof.md` in your repo (the **entry point for your reviewer**):

```markdown
# Lab Proof: Agents that Speak MCP Fluently

## 1. What I built
2-3 lines: which servers, which transport. Link to this notebook.

## 2. Query
"..."

## 3. Evidence
The [LOG] lines from the interceptor: every tool call and what it returned.

## 4. Final answer
(paste the agent's answer)

## 5. Trace
Which claim in the answer comes from which tool call / file?

## 6. Failure or limitation
Where could the system produce a plausible but unsupported answer?
```

**Ideas for the failure:**
- Ask something **not** in the files (*"What is Project Atlas's budget?"*). Does it admit it, or invent a number?
- Ask a question where only **part** of the answer is in the files. Does it mix evidence with guesses?
- Look at the Filesystem tool list: some tools can **write** or **move** files. What could go wrong with client data?

**Before you submit:**
- [ ] No API keys, tokens or `.env` committed
- [ ] Only files for this lab are in the repo, with clear names
- [ ] `lab_proof.md` is the entry point
- [ ] A reviewer can trace **one** answer back to the exact tool call